# Primer ejercicioplanteado por Pecas mi Agente , que trazo la ruta para POO
## Ejemplo
### Un sensor como objeto: el patrón que ya usas en firmware

In [1]:
class SensorTemperatura:
    """Un sensor: estado propio + comportamiento propio."""

    def __init__(self, nombre: str, unidad: str = "°C"):
        self.nombre = nombre
        self.unidad = unidad
        self.lecturas: list[float] = []

    def registrar(self, valor: float) -> None:
        self.lecturas.append(valor)

    def promedio(self) -> float:
        if not self.lecturas:
            return 0.0
        return sum(self.lecturas) / len(self.lecturas)


horno = SensorTemperatura("horno-1")
motor = SensorTemperatura("motor-2")

horno.registrar(180.5)
horno.registrar(182.0)

print(horno.promedio())   # 181.25
print(motor.promedio())   # 0.0  ← estado independiente

181.25
0.0


## Ejercicio 1
> Modela un actuador de tu mundo: una clase BombaAgua (o el dispositivo que prefieras) con estado y comportamiento propios.

1. Define la clase con __init__ que reciba nombre y caudal_l_min, y un atributo encendida que arranque en False.
2. Agrega métodos encender(), apagar() y estado() que devuelva un texto como 'Bomba cocina: APAGADA'.
3. Crea dos instancias distintas, enciende solo una y demuestra con estado() que sus estados son independientes.
4. Extra: llama BombaAgua.encender(tu_instancia) a mano y comprueba que equivale a tu_instancia.encender().

> *Pista*: Si las dos bombas cambian juntas, pusiste el atributo en la clase en vez de en self. Ese error es justo el tema siguiente.



In [3]:
# Parte 1 crear la clase
class BombaAgua:
    # Parte 1.1 definir la clase y atributo
    def __init__(self, nombre:str, caudal_l_min:float):
        self.nombre = nombre
        self.caudal_l_min = caudal_l_min
        # Atributo
        self.encendida:bool = False

    # Parte 2 agregar metodos
    # 2.1 Metodo encender()
    def encender(self):
        self.encendida = True

    # 2.2 Metodo apagar()
    def apagar(self):
        self.encendida = False

    # 2.3 Metodo estado()
    def estado(self) -> str:
        estado = "ENCENDIDA" if self.encendida else "APAGADA"
        return f"Bomba {self.nombre}: {estado}"


# Prueba 3 y 4
# Con 2 instancias independientes
bombaCocina = BombaAgua("cocina", 12.5 )
bombaJardin = BombaAgua("Jardin",25.0)

bombaCocina.encender()

# Prueba estado de las bombas
print(bombaCocina.estado())
print(bombaJardin.estado())

# Prueba 4 lalmada explicita desde la clase y su estado
BombaAgua.encender(bombaJardin)
print(bombaJardin.estado())

Bomba cocina: ENCENDIDA
Bomba Jardin: APAGADA
Bomba Jardin: ENCENDIDA


# Atributos de clase vs. de instancia

## 2 · EJEMPLO

In [2]:
class ModelConfig:
    # Atributo de CLASE: un solo objeto, compartido por todas
    SUPPORTED_FRAMEWORKS: list[str] = ["pytorch", "scikit-learn", "xgboost"]

    def __init__(self, model_name: str, learning_rate: float, epochs: int = 10):
        # Atributos de INSTANCIA: cada config tiene los suyos
        self.model_name = model_name
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.is_trained = False


a = ModelConfig("resnet", 0.01)
b = ModelConfig("bert", 0.001, epochs=20)

print(a.__dict__)
# {'model_name': 'resnet', 'learning_rate': 0.01, 'epochs': 10, 'is_trained': False}
# OJO: SUPPORTED_FRAMEWORKS no está ahí — vive en la clase.

a.SUPPORTED_FRAMEWORKS.append("jax")   # ⚠ modificaste la lista COMPARTIDA
print(b.SUPPORTED_FRAMEWORKS)          # ['pytorch', 'scikit-learn', 'xgboost', 'jax']

{'model_name': 'resnet', 'learning_rate': 0.01, 'epochs': 10, 'is_trained': False}
['pytorch', 'scikit-learn', 'xgboost', 'jax']


# 3 · EJERCICIO
> Caza el bug del atributo compartido y luego sepáralo bien: una clase RegistroDispositivos para tu red de sensores.

1. Escribe una clase con un atributo de clase dispositivos = [] y un método registrar(self, nombre) que haga self.dispositivos.append(nombre). Crea dos registros y comprueba que comparten la lista.
2. Explica en tu respuesta, con tus palabras, por qué pasó (quién es el dueño de la lista y qué hace append).
3. Corrige la clase moviendo la lista a __init__ como atributo de instancia y demuestra que ya no se comparte.
4. Deja en la clase solo lo que sí es compartido: una constante VERSION_PROTOCOLO = '1.2'.
Pista: La diferencia clave no es dónde escribes la línea, sino si asignas (self.x = ...) o modificas (self.x.append(...)).

In [ ]:
class Registrodispositivos:
        #Atributo de mi clase
        dispositivos = []
        def __init__(self, nombre:str):
                self.nombre=nombre

        def registrar(self, nombre:str):
                self.dispositivos.append(nombre)

b = Registrodispositivos("hola")
print(b.__dict__)

a = Registrodispositivos("huevos")
print(a.dispositivos)

b.registrar("tu gfa")   
print(b.dispositivos)
a.registrar("red norte")
print(b.dispositivos)
print(a.dispositivos)

{'nombre': 'hola'}
[]
['tu gfa']
['tu gfa', 'red norte']
['tu gfa', 'red norte']


> Pasa que cuando se genera el atruburto de la clase esta siendo compartida por toda la clase entonces al modiogicarse con el metodo de la clase registar esta modifica a todo el atributo de la clase

## Corrección

In [8]:
class Registrodispositivos:
    VERSION_PROTOCOLO:str = "1.2"  

    def __init__(self, nombre:str):
        self.nombre = nombre
        self.dispositivos:list[str] = []

    def registrar(self, nombre:str):
            self.dispositivos.append(nombre)

r1 = Registrodispositivos("Hola menso")
r2 = Registrodispositivos("hola mundo")


r1.registrar("Buenas nalgas")

print(r1.dispositivos)
print(r1.__dict__)
print(r2.dispositivos)
print(r2.__dict__)
print(Registrodispositivos.VERSION_PROTOCOLO)



['Buenas nalgas']
{'nombre': 'Hola menso', 'dispositivos': ['Buenas nalgas']}
[]
{'nombre': 'hola mundo', 'dispositivos': []}
1.2


# Router de telemetría

- Imagina que estás armando un nodo IoT que recibe paquetes de datos de distintos sensores y los enruta.

> Requerimientos:

1. Atributo de clase (constante compartida):

- Define en la clase una constante MAX_PAQUETES = 3 (el límite máximo de paquetes que soporta el búfer de cualquier nodo).

2. Constructor (__init__):

- Debe recibir únicamente el id_nodo: str (por ejemplo "nodo-cocina").
- Debe inicializar internamente dos cosas en self:
- self.id_nodo = id_nodo
- self.buffer: list[dict] = [] (una lista vacía para almacenar los paquetes de este nodo en particular).

3. Método enviar(self, paquete: dict) -> bool:

- Si la cantidad de paquetes en self.buffer es menor que self.MAX_PAQUETES, agrega el paquete a self.buffer y retorna True.
- Si el búfer ya está lleno (alcanzó MAX_PAQUETES), no agregues el paquete, imprime "Búfer lleno en <id_nodo>" y retorna False.

4. Método vaciar(self) -> list[dict]:

- Guarda los paquetes acumulados, deja self.buffer vacío ([]), y retorna la lista de paquetes que estaban guardados.

5. Prueba que demuestre:

- Creas dos nodos: nodo_a y nodo_b.
- Le envías 3 paquetes al nodo_a y 1 paquete al nodo_b.
- Demuestras que nodo_a se llena y rechaza un 4to paquete, mientras que nodo_b sigue teniendo espacio (sus buffers son 100% independientes).
- Verificas que ambos nodos acceden al mismo MAX_PAQUETES sin duplicarlo.

In [1]:
class Router:
    MAX_PAQUETES =3
    def __init__(self,id:str):
        self.id = id
        self.buffer:list[dict]= []

    def send_pqt(self,paquete:dict) -> bool:
        if len(self.buffer) < self.MAX_PAQUETES:
            self.buffer.append(paquete)
            return True
        else:
            print(f"buffer lleno en {self.id}")
            return False

    def vaciar_pqt(self) -> list[dict]:
        paquetes_backup = self.buffer
        self.buffer = []
        return paquetes_backup

# Crar nodos
nodo_a = Router("Nodo Principal")
nodo_b = Router("Nodo Secundario")

# Envio de 3 paquetes nodo a
nodo_a.send_pqt({"pqt1":123456})
nodo_a.send_pqt({"pqt2":789012})
nodo_a.send_pqt({"pqt3":345678})
print(nodo_a.__dict__)
        
# Envio de un paquete al nodo b
nodo_b.send_pqt({"pqt1":123456})
print(nodo_b.__dict__)

# Muestra nodo lleno 
nodo_a.send_pqt({"pqt4":345678})
nodo_a.send_pqt({"pqt5":345678})

paquetes_extraidos = nodo_a.vaciar_pqt()
print(f"paquetes exatraidos de nodo_a: {paquetes_extraidos}")
print(f"nodo_a despues de paquetes extraidos:{nodo_a.buffer}")



    

{'id': 'Nodo Principal', 'buffer': [{'pqt1': 123456}, {'pqt2': 789012}, {'pqt3': 345678}]}
{'id': 'Nodo Secundario', 'buffer': [{'pqt1': 123456}]}
buffer lleno en Nodo Principal
buffer lleno en Nodo Principal
paquetes exatraidos de nodo_a: [{'pqt1': 123456}, {'pqt2': 789012}, {'pqt3': 345678}]
nodo_a despues de paquetes extraidos:[]


# 4 · EJERCICIO DE REPASO: Gestor de Máquinas y Alertas IoT

> **Objetivo**: Integrar atributos de clase, atributos de instancia (`self`), manejo de listas limpias sin reasignar `.append()`, y formato de estado.

### Contexto
Estás creando un monitor para supervisar máquinas en una planta industrial. Cada máquina tiene su propio registro de eventos, pero todas comparten un límite crítico de advertencias y la nave donde operan.

### Requerimientos:
1. **Atributos de Clase (compartidos por todas las máquinas)**:
   - `MAX_ALERTAS = 2` (número máximo de alertas toleradas antes de parada preventiva).
   - `PLANTA = 'Nave-A'` (nombre general de la instalación).

2. **Constructor (`__init__`)**:
   - Recibe `nombre_maquina: str` y `tipo: str` (ej. `'Torno-1'`, `'CNC'`).
   - Inicializa en `self`:
     - `self.nombre_maquina = nombre_maquina`
     - `self.tipo = tipo`
     - `self.alertas: list[str] = []` (lista vacía de alertas propias de esta máquina).
     - `self.activa: bool = True` (inicia en operación).

3. **Método `reportar_alerta(self, mensaje: str) -> None`**:
   - Si la máquina está activa (`self.activa`):
     - Agrega el mensaje a `self.alertas` (¡recuerda solo llamar a `.append()`, sin reasignar!).
     - Si la cantidad de alertas (`len(self.alertas)`) alcanza o supera `self.MAX_ALERTAS`:
       - Cambia `self.activa = False`.
       - Imprime: `f'¡ALERTA CRÍTICA! Parada preventiva en {self.nombre_maquina}'`.
   - Si la máquina ya estaba inactiva (`not self.activa`):
     - Imprime: `f'{self.nombre_maquina} está detenida. No admite más operaciones.'`

4. **Método `resumen(self) -> str`**:
   - Retorna un texto formateado indicando la planta, el nombre, el estado (`ACTIVA` o `PARADA`), y cuántas alertas tiene acumuladas.
     *Ejemplo*: `'[Nave-A] Torno-1: PARADA (2 alertas)'`

5. **Prueba que debes incluir en la celda de código**:
   - Crea 2 máquinas distintas: `m1` y `m2`.
   - Reporta 1 alerta a `m1` y comprueba que sigue activa.
   - Reporta 2 alertas a `m2` y verifica que se detenga automáticamente.
   - Muestra con `resumen()` que las alertas y estados de `m1` y `m2` son completamente independientes.

In [12]:
'''
1. **Atributos de Clase (compartidos por todas las máquinas)**:
   - `MAX_ALERTAS = 2` (número máximo de alertas toleradas antes de parada preventiva).
   - `PLANTA = 'Nave-A'` (nombre general de la instalación).
'''
class Gestor_Maquinas_Alertas:
    MAX_ALERTAS = 2
    PLANTA = 'Nave-A'
    
    # Inicio del Contructor
    def __init__(self,nombre_maquina:str, tipo:str):
        self.nombre_maquina = nombre_maquina
        self.tipo = tipo
        self.alertas:list[str]= []
        self.activa:bool = True
    # metodo para reportar alertas con maximo de MAX_ALERTAS  
    def reportar_alerta(self,mensaje:str) -> None:
        self.alertas.append(mensaje)
        if len(self.alertas) >= self.MAX_ALERTAS and self.activa:
          self.activa = False
          print(f"¡ALERTA CRITICA! parada preventiva en {self.nombre_maquina}.")
        elif self.activa == False:
          self.alertas.remove(mensaje)
          print(f"{self.nombre_maquina} se encuentra detenida. No permite más operaciones.")

    # Metodo resumen muetra el resumen actual de la maquina
    def resumen(self) -> str:
      if self.activa == True:
        estado:str = "ACTIVA"
      else:
        estado:str = "PARADA"
      return f"[{self.PLANTA}] {self.nombre_maquina} {self.tipo} : {estado} {len(self.alertas)} alertas"

'''
5. **Prueba que debes incluir en la celda de código**:
   - Crea 2 máquinas distintas: `m1` y `m2`.
   - Reporta 1 alerta a `m1` y comprueba que sigue activa.
   - Reporta 2 alertas a `m2` y verifica que se detenga automáticamente.
   - Muestra con `resumen()` que las alertas y estados de `m1` y `m2` son completamente independientes.
'''
m1 = Gestor_Maquinas_Alertas("Perfiladora", "torno")
m2 = Gestor_Maquinas_Alertas("acabados", "fresa")

m1.reportar_alerta("atasco en buril")
m2.reportar_alerta("broca rota, cambiar broca")
m2.reportar_alerta("Biruta atascada")
m2.reportar_alerta("No prende")

print(m1.resumen())
print(m2.resumen())

¡ALERTA CRITICA! parada preventiva en acabados.
acabados se encuentra detenida. No permite más operaciones.
[Nave-A] Perfiladora torno : ACTIVA 1 alertas
[Nave-A] acabados fresa : PARADA 2 alertas
